# Subject: preprocessor
Source: /home/devuser/edu_data/datasets/TrainingDB/EncryT/cpython/Tools/c-analyzer/c_parser/preprocessor

### File: __main__.py

In [ ]:
import logging
import sys

from c_common.scriptutil import (
    add_verbosity_cli,
    add_traceback_cli,
    add_kind_filtering_cli,
    add_files_cli,
    add_failure_filtering_cli,
    add_commands_cli,
    process_args_by_key,
    configure_logger,
    get_prog,
    main_for_filenames,
)
from . import (
    errors as _errors,
    get_preprocessor as _get_preprocessor,
)


FAIL = {
    'err': _errors.ErrorDirectiveError,
    'deps': _errors.MissingDependenciesError,
    'os': _errors.OSMismatchError,
}
FAIL_DEFAULT = tuple(v for v in FAIL if v != 'os')


logger = logging.getLogger(__name__)


##################################
# CLI helpers

def add_common_cli(parser, *, get_preprocessor=_get_preprocessor):
    parser.add_argument('--macros', action='append')
    parser.add_argument('--incldirs', action='append')
    parser.add_argument('--same', action='append')
    process_fail_arg = add_failure_filtering_cli(parser, FAIL)

    def process_args(args, *, argv):
        ns = vars(args)

        process_fail_arg(args, argv=argv)
        ignore_exc = ns.pop('ignore_exc')
        # We later pass ignore_exc to _get_preprocessor().

        args.get_file_preprocessor = get_preprocessor(
            file_macros=ns.pop('macros'),
            file_incldirs=ns.pop('incldirs'),
            file_same=ns.pop('same'),
            ignore_exc=ignore_exc,
            log_err=print,
        )
    return process_args


def _iter_preprocessed(filename, *,
                       get_preprocessor,
                       match_kind=None,
                       pure=False,
                       ):
    preprocess = get_preprocessor(filename)
    for line in preprocess(tool=not pure) or ():
        if match_kind is not None and not match_kind(line.kind):
            continue
        yield line


#######################################
# the commands

def _cli_preprocess(parser, excluded=None, **prepr_kwargs):
    parser.add_argument('--pure', action='store_true')
    parser.add_argument('--no-pure', dest='pure', action='store_const', const=False)
    process_kinds = add_kind_filtering_cli(parser)
    process_common = add_common_cli(parser, **prepr_kwargs)
    parser.add_argument('--raw', action='store_true')
    process_files = add_files_cli(parser, excluded=excluded)

    return [
        process_kinds,
        process_common,
        process_files,
    ]


def cmd_preprocess(filenames, *,
                   raw=False,
                   iter_filenames=None,
                   **kwargs
                   ):
    if 'get_file_preprocessor' not in kwargs:
        kwargs['get_file_preprocessor'] = _get_preprocessor()
    if raw:
        def show_file(filename, lines):
            for line in lines:
                print(line)
                #print(line.raw)
    else:
        def show_file(filename, lines):
            for line in lines:
                linefile = ''
                if line.filename != filename:
                    linefile = f' ({line.filename})'
                text = line.data
                if line.kind == 'comment':
                    text = '/* ' + line.data.splitlines()[0]
                    text += ' */' if '\n' in line.data else r'\n... */'
                print(f' {line.lno:>4} {line.kind:10} | {text}')

    filenames = main_for_filenames(filenames, iter_filenames)
    for filename in filenames:
        lines = _iter_preprocessed(filename, **kwargs)
        show_file(filename, lines)


def _cli_data(parser):
    ...

    return None


def cmd_data(filenames,
             **kwargs
             ):
    # XXX
    raise NotImplementedError


COMMANDS = {
    'preprocess': (
        'preprocess the given C source & header files',
        [_cli_preprocess],
        cmd_preprocess,
    ),
    'data': (
        'check/manage local data (e.g. excludes, macros)',
        [_cli_data],
        cmd_data,
    ),
}


#######################################
# the script

def parse_args(argv=sys.argv[1:], prog=sys.argv[0], *,
               subset='preprocess',
               excluded=None,
               **prepr_kwargs
               ):
    import argparse
    parser = argparse.ArgumentParser(
        prog=prog or get_prog(),
    )

    processors = add_commands_cli(
        parser,
        commands={k: v[1] for k, v in COMMANDS.items()},
        commonspecs=[
            add_verbosity_cli,
            add_traceback_cli,
        ],
        subset=subset,
    )

    args = parser.parse_args(argv)
    ns = vars(args)

    cmd = ns.pop('cmd')

    verbosity, traceback_cm = process_args_by_key(
        args,
        argv,
        processors[cmd],
        ['verbosity', 'traceback_cm'],
    )

    return cmd, ns, verbosity, traceback_cm


def main(cmd, cmd_kwargs):
    try:
        run_cmd = COMMANDS[cmd][0]
    except KeyError:
        raise ValueError(f'unsupported cmd {cmd!r}')
    run_cmd(**cmd_kwargs)


if __name__ == '__main__':
    cmd, cmd_kwargs, verbosity, traceback_cm = parse_args()
    configure_logger(verbosity)
    with traceback_cm:
        main(cmd, cmd_kwargs)

### File: common.py

In [ ]:
import contextlib
import distutils.ccompiler
import logging
import os
import shlex
import subprocess
import sys

from ..info import FileInfo, SourceLine
from .errors import (
    PreprocessorFailure,
    ErrorDirectiveError,
    MissingDependenciesError,
    OSMismatchError,
)


logger = logging.getLogger(__name__)


# XXX Add aggregate "source" class(es)?
#  * expose all lines as single text string
#  * expose all lines as sequence
#  * iterate all lines


def run_cmd(argv, *,
            #capture_output=True,
            stdout=subprocess.PIPE,
            #stderr=subprocess.STDOUT,
            stderr=subprocess.PIPE,
            text=True,
            check=True,
            **kwargs
            ):
    if isinstance(stderr, str) and stderr.lower() == 'stdout':
        stderr = subprocess.STDOUT

    kw = dict(locals())
    kw.pop('argv')
    kw.pop('kwargs')
    kwargs.update(kw)

    # Remove LANG environment variable: the C parser doesn't support GCC
    # localized messages
    env = dict(os.environ)
    env.pop('LANG', None)

    proc = subprocess.run(argv, env=env, **kwargs)
    return proc.stdout


def preprocess(tool, filename, cwd=None, **kwargs):
    argv = _build_argv(tool, filename, **kwargs)
    logger.debug(' '.join(shlex.quote(v) for v in argv))

    # Make sure the OS is supported for this file.
    if (_expected := is_os_mismatch(filename)):
        error = None
        raise OSMismatchError(filename, _expected, argv, error, TOOL)

    # Run the command.
    with converted_error(tool, argv, filename):
        # We use subprocess directly here, instead of calling the
        # distutil compiler object's preprocess() method, since that
        # one writes to stdout/stderr and it's simpler to do it directly
        # through subprocess.
        return run_cmd(argv, cwd=cwd)


def _build_argv(
    tool,
    filename,
    incldirs=None,
    includes=None,
    macros=None,
    preargs=None,
    postargs=None,
    executable=None,
    compiler=None,
):
    if includes:
        includes = tuple(f'-include{i}' for i in includes)
        postargs = (includes + postargs) if postargs else includes

    compiler = distutils.ccompiler.new_compiler(
        compiler=compiler or tool,
    )
    if executable:
        compiler.set_executable('preprocessor', executable)

    argv = None
    def _spawn(_argv):
        nonlocal argv
        argv = _argv
    compiler.spawn = _spawn
    compiler.preprocess(
        filename,
        macros=[tuple(v) for v in macros or ()],
        include_dirs=incldirs or (),
        extra_preargs=preargs or (),
        extra_postargs=postargs or (),
    )
    return argv


@contextlib.contextmanager
def converted_error(tool, argv, filename):
    try:
        yield
    except subprocess.CalledProcessError as exc:
        convert_error(
            tool,
            argv,
            filename,
            exc.stderr,
            exc.returncode,
        )


def convert_error(tool, argv, filename, stderr, rc):
    error = (stderr.splitlines()[0], rc)
    if (_expected := is_os_mismatch(filename, stderr)):
        logger.info(stderr.strip())
        raise OSMismatchError(filename, _expected, argv, error, tool)
    elif (_missing := is_missing_dep(stderr)):
        logger.info(stderr.strip())
        raise MissingDependenciesError(filename, (_missing,), argv, error, tool)
    elif '#error' in stderr:
        # XXX Ignore incompatible files.
        error = (stderr.splitlines()[1], rc)
        logger.info(stderr.strip())
        raise ErrorDirectiveError(filename, argv, error, tool)
    else:
        # Try one more time, with stderr written to the terminal.
        try:
            output = run_cmd(argv, stderr=None)
        except subprocess.CalledProcessError:
            raise PreprocessorFailure(filename, argv, error, tool)


def is_os_mismatch(filename, errtext=None):
    # See: https://docs.python.org/3/library/sys.html#sys.platform
    actual = sys.platform
    if actual == 'unknown':
        raise NotImplementedError

    if errtext is not None:
        if (missing := is_missing_dep(errtext)):
            matching = get_matching_oses(missing, filename)
            if actual not in matching:
                return matching
    return False


def get_matching_oses(missing, filename):
    # OSX
    if 'darwin' in filename or 'osx' in filename:
        return ('darwin',)
    elif missing == 'SystemConfiguration/SystemConfiguration.h':
        return ('darwin',)

    # Windows
    elif missing in ('windows.h', 'winsock2.h'):
        return ('win32',)

    # other
    elif missing == 'sys/ldr.h':
        return ('aix',)
    elif missing == 'dl.h':
        # XXX The existence of Python/dynload_dl.c implies others...
        # Note that hpux isn't actual supported any more.
        return ('hpux', '???')

    # unrecognized
    else:
        return ()


def is_missing_dep(errtext):
    if 'No such file or directory' in errtext:
        missing = errtext.split(': No such file or directory')[0].split()[-1]
        return missing
    return False

### File: errors.py

In [ ]:
import sys


OS = sys.platform


def _as_tuple(items):
    if isinstance(items, str):
        return tuple(items.strip().replace(',', ' ').split())
    elif items:
        return tuple(items)
    else:
        return ()


class PreprocessorError(Exception):
    """Something preprocessor-related went wrong."""

    @classmethod
    def _msg(cls, filename, reason, **ignored):
        msg = 'failure while preprocessing'
        if reason:
            msg = f'{msg} ({reason})'
        return msg

    def __init__(self, filename, preprocessor=None, reason=None):
        if isinstance(reason, str):
            reason = reason.strip()

        self.filename = filename
        self.preprocessor = preprocessor or None
        self.reason = str(reason) if reason else None

        msg = self._msg(**vars(self))
        msg = f'({filename}) {msg}'
        if preprocessor:
            msg = f'[{preprocessor}] {msg}'
        super().__init__(msg)


class PreprocessorFailure(PreprocessorError):
    """The preprocessor command failed."""

    @classmethod
    def _msg(cls, error, **ignored):
        msg = 'preprocessor command failed'
        if error:
            msg = f'{msg} {error}'
        return msg

    def __init__(self, filename, argv, error=None, preprocessor=None):
        exitcode = -1
        if isinstance(error, tuple):
            if len(error) == 2:
                error, exitcode = error
            else:
                error = str(error)
        if isinstance(error, str):
            error = error.strip()

        self.argv = _as_tuple(argv) or None
        self.error = error if error else None
        self.exitcode = exitcode

        reason = str(self.error)
        super().__init__(filename, preprocessor, reason)


class ErrorDirectiveError(PreprocessorFailure):
    """The file hit a #error directive."""

    @classmethod
    def _msg(cls, error, **ignored):
        return f'#error directive hit ({error})'

    def __init__(self, filename, argv, error, *args, **kwargs):
        super().__init__(filename, argv, error, *args, **kwargs)


class MissingDependenciesError(PreprocessorFailure):
    """The preprocessor did not have access to all the target's dependencies."""

    @classmethod
    def _msg(cls, missing, **ignored):
        msg = 'preprocessing failed due to missing dependencies'
        if missing:
            msg = f'{msg} ({", ".join(missing)})'
        return msg

    def __init__(self, filename, missing=None, *args, **kwargs):
        self.missing = _as_tuple(missing) or None

        super().__init__(filename, *args, **kwargs)


class OSMismatchError(MissingDependenciesError):
    """The target is not compatible with the host OS."""

    @classmethod
    def _msg(cls, expected, **ignored):
        return f'OS is {OS} but expected {expected or "???"}'

    def __init__(self, filename, expected=None, *args, **kwargs):
        if isinstance(expected, str):
            expected = expected.strip()

        self.actual = OS
        self.expected = expected if expected else None

        super().__init__(filename, None, *args, **kwargs)

### File: gcc.py

In [ ]:
import os.path
import re

from . import common as _common

# The following C files must not built with Py_BUILD_CORE.
FILES_WITHOUT_INTERNAL_CAPI = frozenset((
    # Modules/
    '_testcapimodule.c',
    '_testlimitedcapi.c',
    '_testclinic_limited.c',
    'xxlimited.c',
    'xxlimited_35.c',
))

# C files in the fhe following directories must not be built with
# Py_BUILD_CORE.
DIRS_WITHOUT_INTERNAL_CAPI = frozenset((
    '_testcapi',            # Modules/_testcapi/
    '_testlimitedcapi',     # Modules/_testlimitedcapi/
))

TOOL = 'gcc'

META_FILES = {
    '<built-in>',
    '<command-line>',
}

# https://gcc.gnu.org/onlinedocs/cpp/Preprocessor-Output.html
# flags:
#  1  start of a new file
#  2  returning to a file (after including another)
#  3  following text comes from a system header file
#  4  following text treated wrapped in implicit extern "C" block
LINE_MARKER_RE = re.compile(r'^# (\d+) "([^"]+)"((?: [1234])*)$')
PREPROC_DIRECTIVE_RE = re.compile(r'^\s*#\s*(\w+)\b.*')
COMPILER_DIRECTIVE_RE = re.compile(r'''
    ^
    (.*?)  # <before>
    (__\w+__)  # <directive>
    \s*
    [(] [(]
    (
        [^()]*
        (?:
            [(]
            [^()]*
            [)]
            [^()]*
         )*
     )  # <args>
    ( [)] [)] )  # <closed>
''', re.VERBOSE)

POST_ARGS = (
    '-pthread',
    '-std=c99',
    #'-g',
    #'-Og',
    #'-Wno-unused-result',
    #'-Wsign-compare',
    #'-Wall',
    #'-Wextra',
    '-E',
)


def preprocess(filename,
               incldirs=None,
               includes=None,
               macros=None,
               samefiles=None,
               cwd=None,
               ):
    if not cwd or not os.path.isabs(cwd):
        cwd = os.path.abspath(cwd or '.')
    filename = _normpath(filename, cwd)

    postargs = POST_ARGS
    basename = os.path.basename(filename)
    dirname = os.path.basename(os.path.dirname(filename))
    if (basename not in FILES_WITHOUT_INTERNAL_CAPI
       and dirname not in DIRS_WITHOUT_INTERNAL_CAPI):
        postargs += ('-DPy_BUILD_CORE=1',)

    text = _common.preprocess(
        TOOL,
        filename,
        incldirs=incldirs,
        includes=includes,
        macros=macros,
        #preargs=PRE_ARGS,
        postargs=postargs,
        executable=['gcc'],
        compiler='unix',
        cwd=cwd,
    )
    return _iter_lines(text, filename, samefiles, cwd)


def _iter_lines(text, reqfile, samefiles, cwd, raw=False):
    lines = iter(text.splitlines())

    # The first line is special.
    # The next two lines are consistent.
    firstlines = [
        f'# 0 "{reqfile}"',
        '# 0 "<built-in>"',
        '# 0 "<command-line>"',
    ]
    if text.startswith('# 1 '):
        # Some preprocessors emit a lineno of 1 for line-less entries.
        firstlines = [l.replace('# 0 ', '# 1 ') for l in firstlines]
    for expected in firstlines:
        line = next(lines)
        if line != expected:
            raise NotImplementedError((line, expected))

    # Do all the CLI-provided includes.
    filter_reqfile = (lambda f: _filter_reqfile(f, reqfile, samefiles))
    make_info = (lambda lno: _common.FileInfo(reqfile, lno))
    last = None
    for line in lines:
        assert last != reqfile, (last,)
        lno, included, flags = _parse_marker_line(line, reqfile)
        if not included:
            raise NotImplementedError((line,))
        if included == reqfile:
            # This will be the last one.
            assert not flags, (line, flags)
        else:
            assert 1 in flags, (line, flags)
        yield from _iter_top_include_lines(
            lines,
            _normpath(included, cwd),
            cwd,
            filter_reqfile,
            make_info,
            raw,
        )
        last = included
    # The last one is always the requested file.
    assert included == reqfile, (line,)


def _iter_top_include_lines(lines, topfile, cwd,
                            filter_reqfile, make_info,
                            raw):
    partial = 0  # depth
    files = [topfile]
    # We start at 1 in case there are source lines (including blank ones)
    # before the first marker line.  Also, we already verified in
    # _parse_marker_line() that the preprocessor reported lno as 1.
    lno = 1
    for line in lines:
        if line == '# 0 "<command-line>" 2' or line == '# 1 "<command-line>" 2':
            # We're done with this top-level include.
            return

        _lno, included, flags = _parse_marker_line(line)
        if included:
            lno = _lno
            included = _normpath(included, cwd)
            # We hit a marker line.
            if 1 in flags:
                # We're entering a file.
                # XXX Cycles are unexpected?
                #assert included not in files, (line, files)
                files.append(included)
            elif 2 in flags:
                # We're returning to a file.
                assert files and included in files, (line, files)
                assert included != files[-1], (line, files)
                while files[-1] != included:
                    files.pop()
                # XXX How can a file return to line 1?
                #assert lno > 1, (line, lno)
            else:
                if included == files[-1]:
                    # It's the next line from the file.
                    assert lno > 1, (line, lno)
                else:
                    # We ran into a user-added #LINE directive,
                    # which we promptly ignore.
                    pass
        elif not files:
            raise NotImplementedError((line,))
        elif filter_reqfile(files[-1]):
            assert lno is not None, (line, files[-1])
            if (m := PREPROC_DIRECTIVE_RE.match(line)):
                name, = m.groups()
                if name != 'pragma':
                    raise Exception(line)
            else:
                line = re.sub(r'__inline__', 'inline', line)
                if not raw:
                    line, partial = _strip_directives(line, partial=partial)
                yield _common.SourceLine(
                    make_info(lno),
                    'source',
                    line or '',
                    None,
                )
            lno += 1


def _parse_marker_line(line, reqfile=None):
    m = LINE_MARKER_RE.match(line)
    if not m:
        return None, None, None
    lno, origfile, flags = m.groups()
    lno = int(lno)
    assert origfile not in META_FILES, (line,)
    assert lno > 0, (line, lno)
    flags = set(int(f) for f in flags.split()) if flags else ()

    if 1 in flags:
        # We're entering a file.
        assert lno == 1, (line, lno)
        assert 2 not in flags, (line,)
    elif 2 in flags:
        # We're returning to a file.
        #assert lno > 1, (line, lno)
        pass
    elif reqfile and origfile == reqfile:
        # We're starting the requested file.
        assert lno == 1, (line, lno)
        assert not flags, (line, flags)
    else:
        # It's the next line from the file.
        assert lno > 1, (line, lno)
    return lno, origfile, flags


def _strip_directives(line, partial=0):
    # We assume there are no string literals with parens in directive bodies.
    while partial > 0:
        if not (m := re.match(r'[^{}]*([()])', line)):
            return None, partial
        delim, = m.groups()
        partial += 1 if delim == '(' else -1  # opened/closed
        line = line[m.end():]

    line = re.sub(r'__extension__', '', line)
    line = re.sub(r'__thread\b', '_Thread_local', line)

    while (m := COMPILER_DIRECTIVE_RE.match(line)):
        before, _, _, closed = m.groups()
        if closed:
            line = f'{before} {line[m.end():]}'
        else:
            after, partial = _strip_directives(line[m.end():], 2)
            line = f'{before} {after or ""}'
            if partial:
                break

    return line, partial


def _filter_reqfile(current, reqfile, samefiles):
    if current == reqfile:
        return True
    if current == '<stdin>':
        return True
    if current in samefiles:
        return True
    return False


def _normpath(filename, cwd):
    assert cwd
    return os.path.normpath(os.path.join(cwd, filename))

### File: pure.py

In [ ]:
from ..source import (
    opened as _open_source,
)
from . import common as _common


def preprocess(lines, filename=None, cwd=None):
    if isinstance(lines, str):
        with _open_source(lines, filename) as (lines, filename):
            yield from preprocess(lines, filename)
        return

    # XXX actually preprocess...
    for lno, line in enumerate(lines, 1):
        kind = 'source'
        data = line
        conditions = None
        yield _common.SourceLine(
            _common.FileInfo(filename, lno),
            kind,
            data,
            conditions,
        )